
# WAN 2.2 — Geração e Extensão de Vídeo a partir de Imagem (PoC)

Este notebook demonstra uma prova de conceito para:
1. Receber uma imagem de entrada
2. Gerar um vídeo inicial usando o modelo **WAN 2.2 Image-to-Video** (dois transformers fp8) via `diffusers`
3. Estender o vídeo iterativamente, usando os **últimos frames** de cada segmento como condicionamento do próximo, mantendo a direção de movimento e a qualidade visual

### Modelos utilizados

Os modelos fp8 repackaged pela Comfy-Org são **~50% menores** que os fp16 originais:

| Componente | Arquivo | Tamanho |
|---|---|---|
| Transformer High-Noise | `wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors` | ~14.3 GB |
| Transformer Low-Noise | `wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors` | ~14.3 GB |
| VAE | `wan_2.1_vae.safetensors` | ~1.5 GB |
| Text Encoder | `umt5_xxl_fp8_e4m3fn_scaled.safetensors` | ~5 GB |

**Referências:**
- [WAN 2.2 GitHub](https://github.com/Wan-Video/Wan2.2)
- [ComfyUI WAN 2.2 Examples](https://comfyanonymous.github.io/ComfyUI_examples/wan22/)
- [Comfy-Org WAN 2.2 Repackaged (HuggingFace)](https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged)
- [WAN Video Extender (ComfyUI)](https://github.com/Granddyser/wan-video-extender)

---
> **Requisito de hardware:** GPU com ≥ 12 GB de VRAM (ex.: RTX 3080/4080) para os modelos fp8. Use `ENABLE_MODEL_OFFLOAD=True` em GPUs com menos VRAM.


## 1. Instalação das Dependências

Instala os pacotes necessários: `torch`, `diffusers`, `transformers`, `accelerate`, `opencv-python`, `imageio`, e `imageio-ffmpeg`.

> Execute esta célula apenas uma vez. Reinicie o kernel após a instalação se necessário.

In [ ]:

# hf-transfer DEVE ser instalado e a variável de ambiente definida ANTES de qualquer
# import do huggingface_hub para que o backend de alta velocidade seja ativado.
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

%pip install --quiet \
    torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 \
    && pip install --quiet \
    "diffusers>=0.33.0" \
    "transformers>=4.45.0" \
    "accelerate>=0.33.0" \
    "opencv-python>=4.10.0" \
    "imageio>=2.35.0" \
    "imageio-ffmpeg>=0.5.1" \
    "numpy>=1.26.0" \
    "Pillow>=10.0.0" \
    "huggingface_hub[cli]>=0.24.0" \
    "hf-transfer>=0.1.8" \
    "safetensors>=0.4.3" \
    "sentencepiece" \
    "protobuf"

print("Instalação concluída. Reinicie o kernel se necessário.")
print("hf-transfer ativo: downloads de modelos serão significativamente mais rápidos.")


## 2. Importação das Bibliotecas

In [ ]:
import gc
import os
from pathlib import Path
from typing import List

import cv2
import imageio
import numpy as np
import torch
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
from diffusers.utils import export_to_video, load_image
from IPython.display import HTML, Video, display
from PIL import Image
from transformers import CLIPVisionModel, UMT5Config, UMT5EncoderModel
import safetensors.torch as sf_torch

print(f"PyTorch versão : {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM total     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 3. Configuração e Parâmetros

Ajuste os parâmetros abaixo conforme o seu hardware e o resultado desejado.

### Modelos WAN 2.2 I2V fp8 (Comfy-Org Repackaged)

O WAN 2.2 usa **dois transformers separados** num pipeline de dois estágios:

| Modelo | Papel | Tamanho |
|---|---|---|
| `wan2.2_i2v_high_noise_14B_fp8_scaled` | Estágio inicial — geração de movimento (timesteps altos) | ~14.3 GB |
| `wan2.2_i2v_low_noise_14B_fp8_scaled` | Estágio final — refinamento de detalhes (timesteps baixos) | ~14.3 GB |

> **Comparação:** O modelo WAN 2.1 bf16 único pesava ~28 GB. Os dois modelos fp8 juntos somam ~28.6 GB, mas cada arquivo é menor e o consumo de VRAM em tempo de execução é cerca de **metade**.

| Parâmetro | Descrição |
|---|---|
| `HF_REPO_ID` | Repositório HuggingFace dos modelos repackaged |
| `MODELS_DIR` | Diretório local para cache dos safetensors |
| `SIGMA_SPLIT` | Limiar σ que separa high-noise de low-noise (~0.4) |
| `INPUT_IMAGE_PATH` | Caminho para a imagem de entrada |
| `PROMPT` | Descrição textual do movimento desejado |
| `NUM_FRAMES` | Frames por segmento de vídeo (padrão: 81) |
| `OVERLAP_FRAMES` | Frames de sobreposição entre segmentos (8–24 recomendado) |
| `EXTENSION_LOOPS` | Número de extensões após o vídeo inicial |
| `FPS` | Taxa de frames por segundo do vídeo exportado |
| `GUIDANCE_SCALE` | Escala de guia CFG (5.0 recomendado) |
| `ENABLE_MODEL_OFFLOAD` | Ativa CPU offload para GPUs com pouca VRAM |


In [ ]:

import os

# Ativa o backend de alta velocidade para downloads do HuggingFace Hub.
# Deve ser definido antes de qualquer import do huggingface_hub.
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# ── Repositório e arquivos dos modelos WAN 2.2 (Comfy-Org Repackaged) ──────────
HF_REPO_ID: str = "Comfy-Org/Wan_2.2_ComfyUI_Repackaged"

# Dois transformers fp8: high-noise para estágio inicial, low-noise para refinamento.
# Cada arquivo pesa ~14.3 GB (metade do equivalente fp16).
HIGH_NOISE_MODEL_FILE: str = "split_files/diffusion_models/wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors"
LOW_NOISE_MODEL_FILE: str  = "split_files/diffusion_models/wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors"

# VAE idêntico ao WAN 2.1 — reutilizável entre versões.
VAE_FILE: str = "split_files/vae/wan_2.1_vae.safetensors"

# Text encoder UMT5-XXL quantizado em fp8 (~5 GB).
TEXT_ENCODER_FILE: str = "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors"

# Image encoder CLIP carregado do repositório HF original do WAN 2.1.
# É um componente pequeno (~1 GB) e não está incluso no Comfy-Org repack.
CLIP_REPO_ID: str = "Wan-AI/Wan2.1-I2V-14B-480P-Diffusers"
CLIP_SUBFOLDER: str = "image_encoder"

# Diretório local para cache dos safetensors baixados
MODELS_DIR = Path("models/wan22")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ── Limiar de sigma para separar os dois estágios de denoising ─────────────────
# Timesteps > SPLIT_TIMESTEP → transformer high-noise (geração de estrutura/movimento)
# Timesteps ≤ SPLIT_TIMESTEP → transformer low-noise  (refinamento de detalhes)
SPLIT_TIMESTEP: int = 500  # equivale a ~σ=0.4 no scheduler Flow-Match

# ── Entrada ─────────────────────────────────────────────────────────────────────
INPUT_IMAGE_PATH: str = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/astronaut.jpg"

# ── Prompt ─────────────────────────────────────────────────────────────────────
PROMPT: str = (
    "An astronaut walks slowly across a barren alien landscape. "
    "Camera follows from behind. Cinematic lighting, high quality, smooth motion."
)
NEGATIVE_PROMPT: str = (
    "Bright tones, overexposed, static, blurred details, subtitles, "
    "worst quality, low quality, deformed, disfigured, still picture, "
    "walking backwards, frozen movement"
)

# ── Parâmetros de geração ───────────────────────────────────────────────────────
NUM_FRAMES: int = 81          # Frames por segmento (WAN usa múltiplos de 4 + 1)
OVERLAP_FRAMES: int = 16      # Frames de contexto reutilizados entre segmentos
EXTENSION_LOOPS: int = 3      # Quantas vezes estender após o vídeo inicial
FPS: int = 16                 # Taxa de frames do vídeo exportado
GUIDANCE_SCALE: float = 5.0   # CFG scale (5.0 recomendado para I2V)
NUM_INFERENCE_STEPS: int = 40 # Passos de difusão (40 é padrão para I2V-14B)
SEED: int = 42                # Semente para reprodutibilidade

# ── Resolução máxima (área) ─────────────────────────────────────────────────────
MAX_AREA: int = 480 * 832

# ── Hardware ────────────────────────────────────────────────────────────────────
DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
ENABLE_MODEL_OFFLOAD: bool = not torch.cuda.is_available() or (
    torch.cuda.get_device_properties(0).total_memory < 16 * 1e9
)

# ── Saída ───────────────────────────────────────────────────────────────────────
OUTPUT_DIR: Path = Path("outputs/wan_poc")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuração carregada:")
print(f"  Repositório    : {HF_REPO_ID}")
print(f"  High-noise     : {HIGH_NOISE_MODEL_FILE.split('/')[-1]}")
print(f"  Low-noise      : {LOW_NOISE_MODEL_FILE.split('/')[-1]}")
print(f"  Dispositivo    : {DEVICE}")
print(f"  CPU Offload    : {ENABLE_MODEL_OFFLOAD}")
print(f"  Split timestep : {SPLIT_TIMESTEP}")
print(f"  Frames/seg     : {NUM_FRAMES}")
print(f"  Overlap        : {OVERLAP_FRAMES}")
print(f"  Loops ext.     : {EXTENSION_LOOPS}")
print(f"  Saída          : {OUTPUT_DIR.resolve()}")



## 4. Carregamento do Pipeline WAN 2.2 I2V (Dois Transformers fp8)

Carrega o pipeline `WanImageToVideoPipeline` com a arquitetura de **dois transformers** do WAN 2.2.

### Arquitetura de Dois Estágios

O WAN 2.2 separa o processo de denoising em dois estágios controlados por `SPLIT_TIMESTEP`:

```
Ruído puro                                     Vídeo final
[t=1000 ──────── t=500] [t=500 ─────────── t=0]
     ↑ transformer_high        ↑ transformer_low
  (estrutura/movimento)     (detalhes/qualidade)
```

A classe `WAN22DualTransformer` intercepta cada chamada do denoiser e roteia
automaticamente para o transformer correto com base no timestep atual.

### Carregamento de Componentes

| Componente | Fonte | Dtype |
|---|---|---|
| `transformer_high` | safetensors local (fp8 → bfloat16 em inferência) | `bfloat16` |
| `transformer_low` | safetensors local (fp8 → bfloat16 em inferência) | `bfloat16` |
| `vae` | safetensors local | `float32` |
| `text_encoder` | safetensors local (fp8) | `bfloat16` |
| `image_encoder` (CLIP) | HuggingFace `Wan-AI/Wan2.1-I2V-14B-480P-Diffusers` | `float32` |
| `tokenizer`, `scheduler` | HuggingFace `Wan-AI/Wan2.1-I2V-14B-480P-Diffusers` | — |


In [ ]:

from huggingface_hub import hf_hub_download


def download_wan22_model_files(
    repo_id: str,
    models_dir: Path,
    high_noise_file: str,
    low_noise_file: str,
    vae_file: str,
    text_encoder_file: str,
) -> dict[str, Path]:
    """
    Baixa os arquivos safetensors necessários para WAN 2.2 I2V fp8.

    Utiliza hf_hub_download com cache local para evitar downloads repetidos.
    O hf-transfer (quando ativo via HF_HUB_ENABLE_HF_TRANSFER=1) realiza
    transferências em paralelo via Rust, acelerando em até 10× downloads grandes.

    Args:
        repo_id: Repositório HuggingFace (ex: "Comfy-Org/Wan_2.2_ComfyUI_Repackaged").
        models_dir: Diretório local para salvar os arquivos.
        high_noise_file: Caminho relativo no repo para o transformer high-noise.
        low_noise_file: Caminho relativo no repo para o transformer low-noise.
        vae_file: Caminho relativo no repo para o VAE.
        text_encoder_file: Caminho relativo no repo para o text encoder.

    Returns:
        Dicionário mapeando nome do componente para caminho local do arquivo.
    """
    files_to_download = {
        "high_noise": high_noise_file,
        "low_noise": low_noise_file,
        "vae": vae_file,
        "text_encoder": text_encoder_file,
    }

    local_paths: dict[str, Path] = {}

    for component_name, repo_filename in files_to_download.items():
        local_filename = Path(repo_filename).name
        local_path = models_dir / local_filename

        if local_path.exists():
            print(f"  [{component_name}] Já existe: {local_path}")
        else:
            print(f"  [{component_name}] Baixando '{local_filename}' do HuggingFace...")
            downloaded = hf_hub_download(
                repo_id=repo_id,
                filename=repo_filename,
                local_dir=str(models_dir),
                local_dir_use_symlinks=False,
            )
            local_path = Path(downloaded)
            print(f"  [{component_name}] Salvo em: {local_path}")

        local_paths[component_name] = local_path

    return local_paths


print("Iniciando download dos modelos WAN 2.2 I2V fp8...")
print(f"Destino: {MODELS_DIR.resolve()}")
print()

model_paths = download_wan22_model_files(
    repo_id=HF_REPO_ID,
    models_dir=MODELS_DIR,
    high_noise_file=HIGH_NOISE_MODEL_FILE,
    low_noise_file=LOW_NOISE_MODEL_FILE,
    vae_file=VAE_FILE,
    text_encoder_file=TEXT_ENCODER_FILE,
)

print()
print("Download concluído:")
for component, path in model_paths.items():
    size_gb = path.stat().st_size / 1e9
    print(f"  {component:<15}: {path.name}  ({size_gb:.2f} GB)")



## 3.5 Download dos Modelos WAN 2.2

Baixa os arquivos `.safetensors` do repositório `Comfy-Org/Wan_2.2_ComfyUI_Repackaged`.

O `hf-transfer` (instalado na célula 1) acelera os downloads em até **10×** usando
transferências paralelas em Rust. Para ativá-lo, a variável de ambiente
`HF_HUB_ENABLE_HF_TRANSFER=1` já foi definida nas células anteriores.

| Arquivo | Tamanho | Uso |
|---|---|---|
| `wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors` | ~14.3 GB | Transformer estágio 1 |
| `wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors` | ~14.3 GB | Transformer estágio 2 |
| `wan_2.1_vae.safetensors` | ~1.5 GB | VAE de decodificação |
| `umt5_xxl_fp8_e4m3fn_scaled.safetensors` | ~5 GB | Text encoder UMT5-XXL |

> Execute esta célula apenas uma vez. Se os arquivos já estiverem em `MODELS_DIR`,
> o `hf_hub_download` retorna o caminho local sem rebaixar.


In [ ]:
import gc
import torch.nn as nn
from diffusers import WanTransformer3DModel


class _TransformerConfigProxy:
    """
    Proxy leve para expor a configuração do transformer como atributos.

    Necessário porque o pipeline acessa `transformer.config.patch_size` e outros
    campos como atributos, não como chaves de dicionário.
    """

    def __init__(self, config_dict: dict) -> None:
        self.__dict__.update(config_dict)

    def __getitem__(self, key: str):
        return self.__dict__[key]

    def get(self, key: str, default=None):
        return self.__dict__.get(key, default)


class WAN22LazyDualTransformer(nn.Module):
    """
    Gerenciador lazy de dois transformers WAN 2.2 com swap sob demanda.

    Resolve o problema de estouro de memória RAM ao garantir que somente
    UM transformer ocupe memória por vez. A troca ocorre tipicamente UMA
    vez por geração (quando o timestep cruza split_timestep), antes da
    qual o transformer ativo é deletado e a memória é liberada.

    Diagrama de uso de memória:
        Fase inicial (t > split)  : transformer_high na GPU, low no disco
        Troca (t cruza threshold) : del high → gc.collect() → carrega low
        Fase final   (t ≤ split)  : transformer_low na GPU, high no disco

    Uso típico:
        RAM pico : ~28 GB (bf16 temporário durante carregamento de cada transformer)
        VRAM ativa: ~28 GB (um transformer bfloat16 por vez)

    Para GPUs com < 30 GB de VRAM, use ENABLE_MODEL_OFFLOAD=True, que
    ativa sequential_cpu_offload nos demais componentes do pipeline.
    O WAN22LazyDualTransformer gerencia sua própria memória independentemente.
    """

    def __init__(
        self,
        high_noise_path: Path,
        low_noise_path: Path,
        transformer_config: "_TransformerConfigProxy",
        split_timestep: int = 500,
        inference_device: str = "cuda",
    ) -> None:
        super().__init__()
        self.high_noise_path = high_noise_path
        self.low_noise_path = low_noise_path
        self.config = transformer_config
        self.split_timestep = split_timestep
        self.inference_device = inference_device

        # Apenas um transformer carregado por vez; inicia vazio
        self._active_key: str | None = None
        self._active_transformer: WanTransformer3DModel | None = None

    @property
    def dtype(self) -> torch.dtype:
        return torch.bfloat16

    @property
    def device(self) -> torch.device:
        return torch.device(self.inference_device)

    def _swap_to(self, key: str) -> WanTransformer3DModel:
        """
        Ativa o transformer solicitado, descarregando o anterior se necessário.

        O carregamento do safetensors fp8 diretamente para bfloat16 tem pico
        de RAM ~28 GB enquanto o arquivo é lido e convertido. Após mover para
        GPU e deletar a variável CPU, a RAM é liberada.

        Args:
            key: "high" para high-noise ou "low" para low-noise.

        Returns:
            Transformer ativo no inference_device.
        """
        if self._active_key == key and self._active_transformer is not None:
            return self._active_transformer

        # Libera o transformer ativo para liberação de RAM/VRAM antes do próximo load
        if self._active_transformer is not None:
            print(f"\n[swap] Descarregando transformer '{self._active_key}' da memória...")
            del self._active_transformer
            self._active_transformer = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            if torch.cuda.is_available():
                free_vram = torch.cuda.mem_get_info()[0] / 1e9
                print(f"[swap] VRAM livre após descarregamento: {free_vram:.1f} GB")

        path = self.high_noise_path if key == "high" else self.low_noise_path
        print(f"[swap] Carregando transformer '{key}' de {path.name} ...")

        # Carrega fp8 → bfloat16 na CPU, depois move para GPU
        # Deletar `cpu_model` logo após o `.to()` libera os ~28 GB de RAM
        cpu_model = WanTransformer3DModel.from_single_file(
            str(path),
            torch_dtype=torch.bfloat16,
        )
        self._active_transformer = cpu_model.to(self.inference_device)
        del cpu_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        self._active_key = key
        print(f"[swap] Transformer '{key}' ativo em '{self.inference_device}'.")
        return self._active_transformer

    def preload(self, key: str = "high") -> None:
        """Pré-carrega o transformer antes da primeira geração para evitar delay."""
        self._swap_to(key)

    def forward(self, hidden_states: torch.Tensor, timestep: torch.Tensor, **kwargs):
        """
        Roteia a chamada para o transformer correto conforme o timestep atual.

        A troca só ocorre quando o key muda (tipicamente uma vez por geração).
        """
        current_t = (
            timestep.flatten()[0].item()
            if isinstance(timestep, torch.Tensor)
            else float(timestep)
        )
        key = "high" if current_t > self.split_timestep else "low"
        transformer = self._swap_to(key)
        return transformer(hidden_states, timestep, **kwargs)


def load_wan22_pipeline(
    model_paths: dict[str, Path],
    clip_repo_id: str,
    clip_subfolder: str,
    base_repo_id: str,
    split_timestep: int,
    enable_offload: bool,
) -> WanImageToVideoPipeline:
    """
    Carrega o pipeline WAN 2.2 I2V com lazy dual transformer.

    O transformer NÃO é carregado aqui — o carregamento ocorre lazily
    na primeira chamada de inferência via `WAN22LazyDualTransformer._swap_to()`.
    Isso garante que nunca mais de um transformer (28 GB) ocupe RAM/VRAM ao mesmo
    tempo durante o carregamento inicial.

    Args:
        model_paths: Caminhos locais dos safetensors (high_noise, low_noise, vae, text_encoder).
        clip_repo_id: Repositório HF do image encoder CLIP.
        clip_subfolder: Subfolder do CLIP no repositório.
        base_repo_id: Repositório base para tokenizer, scheduler e config.
        split_timestep: Timestep (0–1000) que separa high-noise de low-noise.
        enable_offload: Se True, habilita CPU offload para VAE e encoders.

    Returns:
        Pipeline WanImageToVideoPipeline com WAN22LazyDualTransformer.
    """
    # Carrega apenas a configuração JSON do transformer (sem pesos — ~2 KB)
    print("Carregando configuração do transformer (sem pesos)...")
    config_dict = WanTransformer3DModel.load_config(
        base_repo_id,
        subfolder="transformer",
    )
    transformer_config = _TransformerConfigProxy(config_dict)

    # Cria o gerenciador lazy — nenhum transformer é carregado na RAM ainda
    dual_transformer = WAN22LazyDualTransformer(
        high_noise_path=model_paths["high_noise"],
        low_noise_path=model_paths["low_noise"],
        transformer_config=transformer_config,
        split_timestep=split_timestep,
        # Com enable_offload, o device inicial ainda é CUDA; os demais
        # componentes do pipeline ficam em CPU e são movidos sob demanda.
        inference_device=DEVICE,
    )

    print("Carregando VAE (float32)...")
    vae = AutoencoderKLWan.from_single_file(
        str(model_paths["vae"]),
        torch_dtype=torch.float32,
    )

    print(f"Carregando text encoder UMT5-XXL de '{model_paths['text_encoder'].name}' (fp8 → bfloat16)...")
    # Baixa apenas config.json (~2 KB); pesos vêm do safetensors fp8 local.
    # Isso evita ~11 GB de download do texto encoder bf16 do repositório WAN 2.1.
    text_encoder_config = UMT5Config.from_pretrained(base_repo_id, subfolder="text_encoder")
    _fp8_state_dict = sf_torch.load_file(str(model_paths["text_encoder"]), device="cpu")
    # Converte fp8 → bfloat16 tensor por tensor para evitar pico de RAM
    _state_dict_bf16 = {k: v.to(torch.bfloat16) for k, v in _fp8_state_dict.items()}
    del _fp8_state_dict
    gc.collect()
    text_encoder = UMT5EncoderModel(text_encoder_config)
    _missing, _unexpected = text_encoder.load_state_dict(_state_dict_bf16, strict=False)
    del _state_dict_bf16
    if _unexpected:
        print(f"  text_encoder: {len(_unexpected)} chaves inesperadas ignoradas")
    if _missing:
        print(f"  text_encoder: {len(_missing)} chaves ausentes")
    text_encoder = text_encoder.to(torch.bfloat16)
    print("  Text encoder carregado.")

    print(f"Carregando image encoder CLIP de '{clip_repo_id}/{clip_subfolder}'...")
    image_encoder = CLIPVisionModel.from_pretrained(
        clip_repo_id,
        subfolder=clip_subfolder,
        torch_dtype=torch.float32,
    )

    print(f"Montando pipeline a partir de '{base_repo_id}' (tokenizer + scheduler apenas)...")
    pipeline = WanImageToVideoPipeline.from_pretrained(
        base_repo_id,
        transformer=dual_transformer,
        vae=vae,
        text_encoder=text_encoder,
        image_encoder=image_encoder,
        torch_dtype=torch.bfloat16,
    )

    if enable_offload:
        # Ativa offload apenas para VAE e encoders (componentes menores).
        # O transformer é gerenciado manualmente pelo WAN22LazyDualTransformer.
        pipeline.enable_sequential_cpu_offload()
        print("CPU offload sequencial ativado para VAE e encoders.")
    else:
        pipeline.to(DEVICE)
        print(f"Pipeline movido para '{DEVICE}'.")

    # Pré-carrega o transformer high-noise antes da primeira geração
    # para evitar delay no primeiro passo de inferência.
    print("\nPré-carregando transformer high-noise (primeira fase de denoising)...")
    dual_transformer.preload("high")

    return pipeline


print("\nPipeline WAN 2.2 I2V (lazy dual transformer) pronto!")

pipe = load_wan22_pipeline(
    model_paths=model_paths,
    clip_repo_id=CLIP_REPO_ID,
    clip_subfolder=CLIP_SUBFOLDER,
    base_repo_id=CLIP_REPO_ID,
    split_timestep=SPLIT_TIMESTEP,
    enable_offload=ENABLE_MODEL_OFFLOAD,
)


## 5. Carregamento e Pré-processamento da Imagem de Entrada

A imagem é redimensionada para que sua área não ultrapasse `MAX_AREA`, preservando a proporção original. O valor de `mod_value` garante que width e height sejam múltiplos do fator de escala espacial do modelo (necessário para evitar artefatos na geração).

In [ ]:
def preprocess_image(image_path: str, max_area: int, pipeline: WanImageToVideoPipeline) -> Image.Image:
    """
    Carrega e redimensiona a imagem de entrada para o tamanho compatível com o modelo.

    O redimensionamento preserva a proporção (aspect ratio) e garante que width e
    height sejam múltiplos de `mod_value` (fator de escala do VAE × patch size).

    Args:
        image_path: Caminho local ou URL pública para a imagem.
        max_area: Área máxima em pixels (ex.: 480*832 para 480P).
        pipeline: Pipeline WAN usado para extrair o mod_value correto.

    Returns:
        Imagem PIL redimensionada e pronta para uso no pipeline.
    """
    image = load_image(image_path)
    original_width, original_height = image.size

    # mod_value garante compatibilidade com a grade interna do transformer
    mod_value: int = (
        pipeline.vae_scale_factor_spatial * pipeline.transformer.config.patch_size[1]
    )
    aspect_ratio = original_height / original_width

    # Calcula novas dimensões respeitando a área máxima e o alinhamento de grade
    new_height = round(np.sqrt(max_area * aspect_ratio)) // mod_value * mod_value
    new_width = round(np.sqrt(max_area / aspect_ratio)) // mod_value * mod_value

    resized_image = image.resize((new_width, new_height), Image.LANCZOS)

    print(f"Imagem original : {original_width}×{original_height}")
    print(f"Imagem redim.   : {new_width}×{new_height}  (mod_value={mod_value})")
    return resized_image


input_image = preprocess_image(INPUT_IMAGE_PATH, MAX_AREA, pipe)

# Exibe a imagem pré-processada diretamente no notebook
display(input_image)
print("Imagem de entrada carregada.")


## 6. Geração do Vídeo Inicial a partir da Imagem

Executa o pipeline WAN 2.2 I2V com a imagem pré-processada e o prompt de texto.

O `WAN22DualTransformer` redireciona cada passo de denoising automaticamente:
- **Passos 0–19** (timestep ≈ 1000→500): `transformer_high` gera a estrutura de movimento
- **Passos 20–39** (timestep ≈ 500→0): `transformer_low` refina os detalhes visuais

O resultado é uma lista de frames PIL que são exportados como arquivo `.mp4`.


In [ ]:

def generate_video_from_image(
    pipeline: WanImageToVideoPipeline,
    image: Image.Image,
    prompt: str,
    negative_prompt: str,
    num_frames: int,
    guidance_scale: float,
    num_inference_steps: int,
    seed: int,
    output_path: Path,
) -> List[Image.Image]:
    """
    Gera um segmento de vídeo a partir de uma imagem usando o pipeline WAN 2.2 I2V.

    O WAN22DualTransformer interno direciona os passos de denoising automaticamente
    entre os transformers high-noise e low-noise sem configuração adicional.

    Args:
        pipeline: Pipeline WanImageToVideoPipeline com dual transformer WAN 2.2.
        image: Imagem PIL de condicionamento (primeiro frame do vídeo).
        prompt: Descrição textual do movimento e conteúdo desejados.
        negative_prompt: Elementos a serem evitados na geração.
        num_frames: Número de frames a gerar neste segmento.
        guidance_scale: Escala CFG (5.0 recomendado para I2V).
        num_inference_steps: Passos de difusão reversa.
        seed: Semente para geração determinística.
        output_path: Caminho para salvar o vídeo exportado.

    Returns:
        Lista de frames PIL gerados.
    """
    generator = torch.Generator(device=DEVICE).manual_seed(seed)

    print(f"Gerando {num_frames} frames (steps={num_inference_steps}, cfg={guidance_scale})...")
    output = pipeline(
        image=image,
        prompt=prompt,
        negative_prompt=negative_prompt,
        height=image.height,
        width=image.width,
        num_frames=num_frames,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
        generator=generator,
    )

    frames: List[Image.Image] = output.frames[0]

    export_to_video(frames, str(output_path), fps=FPS)
    print(f"Segmento salvo em: {output_path}")

    return frames


# Gera o primeiro segmento de vídeo com o pipeline WAN 2.2
initial_video_path = OUTPUT_DIR / "segment_000.mp4"
initial_frames = generate_video_from_image(
    pipeline=pipe,
    image=input_image,
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    num_frames=NUM_FRAMES,
    guidance_scale=GUIDANCE_SCALE,
    num_inference_steps=NUM_INFERENCE_STEPS,
    seed=SEED,
    output_path=initial_video_path,
)

print(f"\nVídeo inicial gerado com {len(initial_frames)} frames.")
print(f"Transformer split em t={SPLIT_TIMESTEP}: high-noise → low-noise @ passo {NUM_INFERENCE_STEPS // 2}/{NUM_INFERENCE_STEPS}")


## 7. Extração dos Últimos Frames do Vídeo

Para manter continuidade entre segmentos, extraímos os **últimos N frames** do vídeo anterior e usamos o **último frame** como imagem de condicionamento do próximo segmento.

O número de frames de sobreposição (`OVERLAP_FRAMES`) controla a consistência:
- **8–12 frames**: transições rápidas, menor consistência visual
- **16–20 frames**: equilíbrio entre velocidade e qualidade (recomendado)
- **24–32 frames**: alta consistência, mas segmentos efetivos mais curtos

In [ ]:
def extract_last_frames(
    frames: List[Image.Image],
    overlap_count: int,
) -> List[Image.Image]:
    """
    Extrai os últimos N frames de uma sequência de frames PIL.

    Esses frames são usados como contexto de sobreposição para a próxima geração,
    garantindo continuidade visual e de movimento entre segmentos.

    Args:
        frames: Lista de frames PIL do segmento atual.
        overlap_count: Número de frames a extrair do final da lista.

    Returns:
        Lista com os últimos `overlap_count` frames.

    Raises:
        ValueError: Se `overlap_count` for maior que o número de frames disponíveis.
    """
    if overlap_count > len(frames):
        raise ValueError(
            f"overlap_count ({overlap_count}) não pode ser maior que o número de "
            f"frames disponíveis ({len(frames)})."
        )
    return frames[-overlap_count:]


def get_last_frame_as_image(frames: List[Image.Image]) -> Image.Image:
    """
    Retorna o último frame de uma lista como imagem PIL.

    Este frame é usado como a imagem de condicionamento (primeiro frame) do
    próximo segmento de vídeo, criando a ilusão de continuidade.

    Args:
        frames: Lista de frames PIL.

    Returns:
        Último frame como objeto PIL.Image.
    """
    return frames[-1].copy()


# Demonstra a extração nos frames do segmento inicial
overlap_frames = extract_last_frames(initial_frames, OVERLAP_FRAMES)
next_seed_image = get_last_frame_as_image(initial_frames)

print(f"Total de frames no segmento inicial: {len(initial_frames)}")
print(f"Frames de sobreposição extraídos   : {len(overlap_frames)}")
print("Último frame (imagem condicionadora):")
display(next_seed_image)

## 8. Extensão de Vídeo a partir dos Últimos Frames

A lógica de extensão usa o **último frame** como imagem de condicionamento.  
Para garantir que o movimento continue na mesma direção, o prompt descreve a continuação da ação, e a sobreposição de frames fornece contexto visual ao modelo.

In [ ]:
def extend_video_segment(
    pipeline: WanImageToVideoPipeline,
    seed_image: Image.Image,
    prompt: str,
    negative_prompt: str,
    num_frames: int,
    overlap_frames_count: int,
    guidance_scale: float,
    num_inference_steps: int,
    seed: int,
    output_path: Path,
) -> List[Image.Image]:
    """
    Gera um segmento de extensão usando o último frame do segmento anterior.

    A mesma API de geração I2V é usada, mas agora a "imagem de entrada" é o
    último frame do vídeo anterior. Isso garante que o modelo continue o
    movimento a partir do estado visual atual da cena.

    Os primeiros `overlap_frames_count` frames do resultado são descartados ao
    montar a sequência final, pois representam a transição de sobreposição.

    Args:
        pipeline: Pipeline WAN I2V carregado.
        seed_image: Último frame do segmento anterior (imagem condicionadora).
        prompt: Descrição textual da continuação do movimento.
        negative_prompt: Elementos a serem evitados.
        num_frames: Total de frames a gerar neste segmento.
        overlap_frames_count: Frames de sobreposição a descartar na concatenação.
        guidance_scale: Escala CFG.
        num_inference_steps: Passos de difusão.
        seed: Semente para variação controlada entre segmentos.
        output_path: Caminho para salvar o segmento exportado.

    Returns:
        Todos os frames gerados (incluindo a sobreposição).
    """
    generator = torch.Generator(device=DEVICE).manual_seed(seed)

    print(
        f"Estendendo vídeo: {num_frames} frames, overlap={overlap_frames_count}, "
        f"seed={seed}..."
    )
    output = pipeline(
        image=seed_image,
        prompt=prompt,
        negative_prompt=negative_prompt,
        height=seed_image.height,
        width=seed_image.width,
        num_frames=num_frames,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
        generator=generator,
    )

    frames: List[Image.Image] = output.frames[0]
    export_to_video(frames, str(output_path), fps=FPS)
    print(f"Segmento de extensão salvo em: {output_path}")

    return frames

## 9. Loop de Extensão do Vídeo

Executa `EXTENSION_LOOPS` iterações, encadeando segmentos de vídeo.  
A cada iteração:
1. O último frame do segmento anterior vira a imagem de entrada
2. Um novo segmento é gerado com o pipeline I2V
3. Os frames de sobreposição do início do novo segmento são descartados na montagem final
4. A semente é incrementada para introduzir variação controlada e evitar loops repetitivos

In [ ]:
# Coleção de todos os segmentos gerados (frames PIL por segmento)
# Cada entrada é (frames_do_segmento, caminho_do_arquivo_mp4)
all_segments: List[tuple[List[Image.Image], Path]] = [
    (initial_frames, initial_video_path)
]

# Frames do segmento atual para extrair a imagem condicionadora
current_frames: List[Image.Image] = initial_frames

for loop_index in range(1, EXTENSION_LOOPS + 1):
    print(f"\n{'='*60}")
    print(f"Loop de extensão {loop_index}/{EXTENSION_LOOPS}")
    print(f"{'='*60}")

    # Usa o último frame do segmento anterior como imagem de entrada
    seed_image = get_last_frame_as_image(current_frames)

    # Incrementa a semente para variação controlada entre segmentos
    current_seed = SEED + loop_index

    segment_path = OUTPUT_DIR / f"segment_{loop_index:03d}.mp4"

    extended_frames = extend_video_segment(
        pipeline=pipe,
        seed_image=seed_image,
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        num_frames=NUM_FRAMES,
        overlap_frames_count=OVERLAP_FRAMES,
        guidance_scale=GUIDANCE_SCALE,
        num_inference_steps=NUM_INFERENCE_STEPS,
        seed=current_seed,
        output_path=segment_path,
    )

    all_segments.append((extended_frames, segment_path))
    current_frames = extended_frames

    # Libera memória de GPU após cada geração para evitar OOM em loops longos
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    new_frames_count = len(extended_frames) - OVERLAP_FRAMES
    print(f"Novos frames gerados (excl. overlap): {new_frames_count}")

print(f"\nTotal de segmentos gerados: {len(all_segments)}")
total_unique_frames = len(initial_frames) + sum(
    len(seg) - OVERLAP_FRAMES for seg, _ in all_segments[1:]
)
print(f"Total de frames únicos estimado    : {total_unique_frames}")
print(f"Duração estimada                   : {total_unique_frames / FPS:.1f}s @ {FPS}fps")

## 10. Concatenação dos Segmentos de Vídeo

Os segmentos são concatenados em um único vídeo final.  
Os frames de **sobreposição** (`OVERLAP_FRAMES`) do início de cada segmento de extensão são **descartados** para evitar frames duplicados na junção.

A concatenação é feita diretamente em memória com `imageio`, ou via **FFmpeg** (mais eficiente para sequências longas).

In [ ]:
def concatenate_video_segments(
    segments: List[tuple[List[Image.Image], Path]],
    overlap_count: int,
    output_path: Path,
    fps: int,
) -> Path:
    """
    Concatena todos os segmentos de vídeo em um único arquivo final.

    O primeiro segmento é incluído integralmente. Para os segmentos subsequentes,
    os primeiros `overlap_count` frames são descartados, pois duplicam frames já
    presentes no segmento anterior (zona de transição).

    Esta abordagem é eficiente em memória: os frames PIL são convertidos para
    arrays NumPy incrementalmente e escritos via imageio/ffmpeg sem acumular
    tudo na RAM simultaneamente.

    Args:
        segments: Lista de tuplas (frames_PIL, caminho_mp4) por segmento.
        overlap_count: Frames de sobreposição a ignorar no início de cada extensão.
        output_path: Caminho para o vídeo final concatenado.
        fps: Taxa de frames do vídeo de saída.

    Returns:
        Caminho do arquivo de vídeo gerado.
    """
    print(f"Concatenando {len(segments)} segmentos → {output_path}")

    with imageio.get_writer(str(output_path), fps=fps, codec="libx264", quality=8) as writer:
        for segment_index, (frames, segment_path) in enumerate(segments):
            # O primeiro segmento é incluído completo; os demais ignoram o overlap
            start_frame = 0 if segment_index == 0 else overlap_count

            frames_written = 0
            for frame in frames[start_frame:]:
                writer.append_data(np.array(frame))
                frames_written += 1

            print(f"  Segmento {segment_index:03d}: {frames_written} frames escritos")

    print(f"\nVídeo final salvo em: {output_path}")
    return output_path


def concatenate_segments_with_ffmpeg(
    segment_paths: List[Path],
    overlap_count: int,
    output_path: Path,
    fps: int,
) -> Path:
    """
    Alternativa baseada em FFmpeg para concatenar segmentos já salvos em disco.

    Mais eficiente para sequências longas pois não carrega todos os frames na RAM.
    Requer que `ffmpeg` esteja instalado no sistema ou via `imageio-ffmpeg`.

    Args:
        segment_paths: Lista de caminhos .mp4 por segmento ordenados.
        overlap_count: Frames a remover do início de cada segmento (exceto o 1º).
        output_path: Caminho para o vídeo de saída.
        fps: Taxa de frames do vídeo de saída.

    Returns:
        Caminho do arquivo de vídeo gerado.
    """
    import subprocess
    import tempfile

    # Cria um arquivo de lista de segmentos temporário para o ffmpeg concat demuxer
    with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as concat_list:
        list_path = concat_list.name

        for segment_index, seg_path in enumerate(segment_paths):
            if segment_index == 0:
                concat_list.write(f"file '{seg_path.resolve()}'\n")
            else:
                # Calcula o timestamp de início para pular os frames de sobreposição
                start_time = overlap_count / fps
                concat_list.write(
                    f"file '{seg_path.resolve()}'\n"
                    f"inpoint {start_time:.4f}\n"
                )

    ffmpeg_cmd = [
        "ffmpeg", "-y",
        "-f", "concat",
        "-safe", "0",
        "-i", list_path,
        "-c:v", "libx264",
        "-crf", "18",
        "-preset", "fast",
        str(output_path),
    ]

    print(f"Executando FFmpeg para concatenar {len(segment_paths)} segmentos...")
    result = subprocess.run(ffmpeg_cmd, capture_output=True, text=True)

    os.unlink(list_path)  # Remove o arquivo temporário

    if result.returncode != 0:
        raise RuntimeError(f"FFmpeg falhou:\n{result.stderr}")

    print(f"Vídeo final salvo em: {output_path}")
    return output_path


# ── Concatenação via imageio (não requer ffmpeg externo) ──────────────────────
final_video_path = OUTPUT_DIR / "final_extended_video.mp4"

final_path = concatenate_video_segments(
    segments=all_segments,
    overlap_count=OVERLAP_FRAMES,
    output_path=final_video_path,
    fps=FPS,
)

print(f"\nVídeo final: {final_path}")

## 11. Exibição e Salvamento do Vídeo Final

Exibe o vídeo final diretamente no notebook usando um player HTML5 embutido e imprime um resumo das estatísticas da geração.

In [ ]:
def display_video(video_path: Path, width: int = 640) -> None:
    """
    Exibe um arquivo de vídeo inline no Jupyter Notebook usando um player HTML5.

    Args:
        video_path: Caminho para o arquivo .mp4.
        width: Largura do player em pixels.
    """
    if not video_path.exists():
        print(f"Arquivo de vídeo não encontrado: {video_path}")
        return

    file_size_mb = video_path.stat().st_size / 1e6

    # Usa IPython.display.Video para renderizar o player embutido
    display(Video(str(video_path), embed=True, width=width))
    print(f"Arquivo: {video_path}")
    print(f"Tamanho: {file_size_mb:.2f} MB")


def print_generation_summary(
    segments: List[tuple[List[Image.Image], Path]],
    overlap_count: int,
    fps: int,
    final_path: Path,
) -> None:
    """
    Imprime um resumo detalhado de todas as etapas de geração.

    Args:
        segments: Lista de segmentos (frames, caminho).
        overlap_count: Frames de sobreposição utilizados.
        fps: Taxa de frames.
        final_path: Caminho do vídeo final concatenado.
    """
    print("=" * 60)
    print("RESUMO DA GERAÇÃO")
    print("=" * 60)

    total_unique = 0
    for index, (frames, seg_path) in enumerate(segments):
        unique = len(frames) if index == 0 else len(frames) - overlap_count
        total_unique += unique
        duration = unique / fps
        print(
            f"  Segmento {index:03d}: {len(frames):4d} frames totais | "
            f"{unique:4d} únicos | {duration:.1f}s"
        )

    total_duration = total_unique / fps
    print("-" * 60)
    print(f"  Total de frames únicos: {total_unique}")
    print(f"  Duração total         : {total_duration:.1f}s @ {fps}fps")

    if final_path.exists():
        file_size_mb = final_path.stat().st_size / 1e6
        print(f"  Tamanho do arquivo    : {file_size_mb:.2f} MB")

    print("=" * 60)


# Exibe o resumo
print_generation_summary(all_segments, OVERLAP_FRAMES, FPS, final_video_path)

# Reproduz o vídeo final no notebook
print("\nVídeo final:")
display_video(final_video_path)

---

## Bônus: Visualização dos Segmentos Individuais

Exibe cada segmento gerado separadamente para facilitar a inspeção visual da qualidade das transições entre loops.

In [ ]:
for segment_index, (_, segment_path) in enumerate(all_segments):
    label = "Vídeo inicial" if segment_index == 0 else f"Extensão {segment_index}"
    print(f"\n{label} ({segment_path.name}):")
    display_video(segment_path, width=480)